### Udaplay_02_solution_project — load, process, and format the provided game JSON files.

This solution 
1. loads, processes, and formats the games.json file
2. creates an agent with necessary tools
3. makes some queries
4. displays the result, how the agent's thinking process was

## 1 Setting up dependencies

import the necessary packages and libraries and load environment

In [1]:
from dotenv import load_dotenv
import os

from lib.agents import Agent
from lib.messages import BaseMessage
from typing import List

load_dotenv("config.env")
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

# 1.1 Create printer function

implement message printer private function to better visualize thinking process and output of the agent

In [2]:
def _print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(
            f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})"
        )

# 2 RAG, Tavily Client, and Long Term Memory initialization


In [3]:
import json
from difflib import SequenceMatcher
from lib.tooling import tool
from lib.vector_db import VectorStore
from tavily import TavilyClient
from chromadb.api.types import QueryResult
from typing import Dict
from datetime import datetime
from lib.llm import LLM
from lib.rag import RAG
import os
from lib.vector_db import VectorStoreManager
from lib.documents import Document
from lib.tooling import Tool
from lib.memory import LongTermMemory, MemoryFragment

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

rag_llm = LLM(model="gpt-4o-mini", temperature=0.3)
vector_store_manager = VectorStoreManager(os.getenv("OPENAI_API_KEY"))
long_term_memory = LongTermMemory(vector_store_manager)

@tool
def get_games(num_games:int=1, top:bool=True) -> str:
    """
    Returns the top or bottom N games with highest or lowest scores.    
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    with open('games.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        # Sort the games list by Score
        # If top is True, descending order
        sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)

        # Return the N games
        return str(sorted_games[:num_games])
    
long_term_memory.vector_store.add(Document(content=get_games(num_games=10, top=True)))
games_rag = RAG(llm=rag_llm, vector_store=long_term_memory.vector_store)

# 3 Define tools to be used by the agent

In [5]:
@tool
def retrieve_game(game_name:str) -> QueryResult:
    """
    Retrieves game information from the vector store based on the game name.
    args:
        game_name (str): The name of the game to retrieve
    returns:
        QueryResult: The result of the query
    """
    return games_rag.invoke(game_name).get_final_state()["answer"]

@tool
def evaluate_retrieval(found_game_name:str, expected_game_name:str) -> float:
    """
    Evaluates the retrieval result by comparing the retrieved game name with the expected game name.
    args:
        found_game_name (str): The result of the retrieval query
        expected_game_name (str): The expected game name to compare against
    returns:
        float: how similar the retrieved game name is to the expected game name in percentage
    """
    if not found_game_name:
        return 0.0  # No documents retrieved, so similarity is 0%
    

    similarity = SequenceMatcher(
        None,
        found_game_name,
        expected_game_name
    ).ratio()
    return round(similarity * 100, 2)

@tool
def game_web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    
    # Perform the search
    search_result = tavily_client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

def build_memory_registration_tool(ltm:LongTermMemory, owner:str, namespace:str):
    """
    Create a tool for agents to register new memories.
    
    This factory function creates a tool that allows AI agents to store new
    information about users in the long-term memory system. The tool is
    pre-configured with specific owner and namespace parameters.
    
    Args:
        ltm (LongTermMemory): The memory system instance to use
        owner (str): User identifier for memory ownership
        namespace (str): Namespace for organizing memories
        
    Returns:
        Tool: A configured tool for memory registration
    """
    def _register(content:str):
        ltm.register(
            MemoryFragment(
                content=content, 
                owner=owner,
                namespace=namespace
            )
        )
        return "Saved new memory"

    return Tool(
        func=_register, 
        name="register_memory", 
        description=(
            "Register a new memory about the information of the game for future reference. "
            "Args:\n"
            "    content: The information to save"
        )
    )

def build_memory_search_tool(ltm:LongTermMemory, owner:str, namespace:str):
    """
    Create a tool for agents to search existing memories.
    
    This factory function creates a tool that allows AI agents to retrieve
    relevant memories from the long-term memory system based on semantic
    similarity to a search query.
    
    Args:
        ltm (LongTermMemory): The memory system instance to use
        owner (str): User identifier for memory ownership
        namespace (str): Namespace to search within
        
    Returns:
        Tool: A configured tool for memory search
    """
    def _search(content:str):
        result = ltm.search(
            query_text=content,
            owner=owner,
            namespace=namespace,
            limit=3,
        )
        return str(tuple(zip(result.fragments, result.metadata['distances'])))

    return Tool(
        func=_search, 
        name="search_memory", 
        description=(
            "Search for a stored memory or preference about the game, " 
            "so it's useful as a context.\n"
            "Args:\n"
            "    content: The information to look for"
        )
    )

# 4 Define a simple agent

Define the agent with necessary tools to look through the RAG first, then evaluate the found answer, if the answer is not satisfactory fall back to long term memory first, if still not found, fall back to web search tool. There are also tools to look through the long term memory and save a new memory to the long term memory

In [6]:
my_agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        build_memory_search_tool(long_term_memory, "omer_oksuzler", "games"),
        game_web_search,
        build_memory_registration_tool(long_term_memory, "omer_oksuzler", "games"),
    ],
    temperature=0.3,
    instructions=(
        "You are a helpful assistant for game recommendations and information retrieval. "
        "When answering a query about a game, you MUST follow this exact tool-use order:\n"
        "1. Call retrieve_game first to search the local game database.\n"
        "1.1 if a game is found, call evaluate_retrieval to check if the retrieved game information is relevant and sufficient to answer the user's query. If the evaluation score is above 70%, you can use this information to answer the query without calling any more tools.\n"
        "2. Only if retrieve_game returns no result or insufficient information, call the long-term memory search tool to check previously stored knowledge.\n"
        "3. Only if the long-term memory search also returns no useful result, call game_web_search to find the information online.\n"
        "4. After using game_web_search, you MUST always call the long-term memory registration tool to persist the retrieved information for future use.\n"
        "Never skip steps or jump ahead. Always incorporate all retrieved information into your final response. Be concise and informative."
    ),
)

# 5 Invoke the agent and print results

Create runs that invoke the agent and prints the result, thinking process, and tool usage

In [7]:
run1 = my_agent.invoke(
    query="give me score and platform info about Evolve", session_id="session_1"
)
run2 = my_agent.invoke(
    query="give me score and platform info about Metroid", session_id="session_2"
)
run3 = my_agent.invoke(
    query="give me score and platform info about Evolve", session_id="session_2"
)
run4 = my_agent.invoke(
    query="give me score and platform info about Evolve", session_id="session_1"
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 3:")
messages = run3.get_final_state()["messages"]
_print_messages(messages)

print("\nMessages from run 4:")
messages = run4.get_final_state()["messages"]
_print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: gener